In [1]:
import deeplabcut
import cv2 as cv
import os
import matplotlib.pyplot as plt
import numpy as np
import math
import sys
import time
import pandas as pd
from pathlib import Path

from fly_tracker_functions import *

Loading DLC 2.3.8...


In [2]:
plate_config_path = os.path.join('plate_finder_curtain-kinglab-2024-03-21',
                                 'config_kmm_mac.yaml')
dlc_plate_string = 'DLC_resnet50_plate_finder_curtainMar21shuffle1_500000'

vidFile = '2023-10-31_CON1_rep2_run1.avi'
vidPath = os.path.join('plate_finder_curtain-kinglab-2024-03-21',
                       'videos',
                       vidFile)

In [3]:
# Plate coordinates
pos = list(range(0, 34))
x1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
      600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600,
      600, 600, 600]
y1 = [24, 79, 133, 187, 241, 297, 355, 413, 469, 524, 577, 634, 689, 744,
      799, 855, 908, 24, 79, 133, 187, 241, 297, 355, 413, 469, 524, 577,
      634, 689, 744, 799, 855, 908]
x2 = [599, 599, 599, 599, 599, 599, 599, 599, 599, 599, 599, 599, 599, 599,
      599, 599, 599, 1199, 1199, 1199, 1199, 1199, 1199, 1199, 1199, 1199,
      1199, 1199, 1199, 1199, 1199, 1199, 1199, 1199]
y2 = [78, 132, 186, 240, 296, 354, 412, 468, 523, 576, 633, 688, 743, 798,
      854, 907, 968, 78, 132, 186, 240, 296, 354, 412, 468, 523, 576, 633,
      688, 743, 798, 854, 907, 968]

pl = pd.DataFrame({'pos': pos, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})
pl['h'] = pl['y2'] - pl['y1']
pl['w'] = pl['x2'] - pl['x1']

In [4]:
splVidPath = os.path.join('training_data', vidFile[:-4] + '_splitvideos')
print('Videos will write into: ' + splVidPath)

# Make directory for split videos
try:
    os.mkdir(splVidPath)
except OSError as error:
    print(error)

Videos will write into: training_data/2023-10-31_CON1_rep2_run1_splitvideos
[Errno 17] File exists: 'training_data/2023-10-31_CON1_rep2_run1_splitvideos'


In [5]:
# Find corners of plate
deeplabcut.analyze_videos(plate_config_path,
                          vidPath,
                          videotype='.avi',
                          save_as_csv=False)


Using snapshot-500000 for model plate_finder_curtain-kinglab-2024-03-21/dlc-models/iteration-0/plate_finder_curtainMar21-trainset95shuffle1


/Users/kmm/miniconda/envs/DEEPLABCUT_M1/lib/python3.9/site-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
2024-03-22 19:00:29.972341: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


Starting to analyze %  plate_finder_curtain-kinglab-2024-03-21/videos/2023-10-31_CON1_rep2_run1.avi
Loading  plate_finder_curtain-kinglab-2024-03-21/videos/2023-10-31_CON1_rep2_run1.avi
Duration of video [s]:  505.0 , recorded with  1.0 fps!
Overall # of frames:  505  found with (before cropping) frame dimensions:  1024 768
Starting to extract posture


100%|██████████| 505/505 [00:50<00:00, 10.08it/s]
/Users/kmm/miniconda/envs/DEEPLABCUT_M1/lib/python3.9/site-packages/deeplabcut/utils/auxiliaryfunctions.py:402: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  DataMachine.to_hdf(dataname, "df_with_missing", format="table", mode="w")


Saving results in plate_finder_curtain-kinglab-2024-03-21/videos...
The videos are analyzed. Now your research can truly start! 
 You can create labeled videos with 'create_labeled_video'
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.


'DLC_resnet50_plate_finder_curtainMar21shuffle1_500000'

In [6]:
# Write movie with plate corners
# time.sleep(10)
# deeplabcut.create_labeled_video(config = plate_config_path,
#                                 videos = [vidPath],
#                                 videotype = '.avi')

In [7]:
# Rename plate corner finder output files
old_file =  os.path.join(vidPath[:-4] + dlc_plate_string + '.h5')
os.rename(old_file, old_file.replace(dlc_plate_string, ''))

# old_file = os.path.join(vidPath[:-4] + dlc_plate_string + '_labeled.mp4')
# os.rename(old_file, old_file.replace(dlc_plate_string, ''))

# Remove pickle created by DLC
os.remove(os.path.join(vidPath[:-4] + dlc_plate_string + '_meta.pickle'))


In [8]:
# Create videoWriters
for vw_id in range(34):
    outfile = os.path.join(splVidPath, vidFile[:-4] + '_' + str(vw_id) + '.mp4')
    globals()['vW{}'.format(vw_id)] = cv.VideoWriter(
        filename=outfile,
        apiPreference=cv.CAP_FFMPEG,
        fourcc=cv.VideoWriter_fourcc('m', 'p', '4', 'v'),
        fps=4,
        frameSize=(pl.w[vw_id], pl.h[vw_id]))

In [9]:
# Processing video ###############################################
# Load video
vid = cv.VideoCapture(vidPath)
nframes = int(vid.get(cv.CAP_PROP_FRAME_COUNT))

# Load csv with tracked corners
corners = pd.read_hdf(vidPath[:-4] + '.h5')
corners = corners[dlc_plate_string]

# Extract coordinates
UL = corners['topleft']
UR = corners['topright']
LL = corners['bottomleft']
LR = corners['bottomright']

In [10]:
# Iterate through frames
print("\nSplitting into separate videos and cropping.\n")
for ii in range(nframes):

    # Load image
    ret, img = vid.read()

    # Corner points
    corner_pts = np.array([
        [LL.iloc[ii, 0], LL.iloc[ii, 1]],
        [UL.iloc[ii, 0], UL.iloc[ii, 1]],
        [UR.iloc[ii, 0], UR.iloc[ii, 1]],
        [LR.iloc[ii, 0], LR.iloc[ii, 1]]
        ])

    # Use four point transformation
    warped = four_point_transform(img, corner_pts)

    # Resize to 1200 x 1000
    dim = (1200, 1000)

    # resize image
    resized = cv.resize(warped, dim, interpolation = cv.INTER_AREA)

    # Plate segmentation check. Use 100th frame.
    if ii == 99:
        seg_check = resized.copy()
        for idx, row in pl.iterrows():
            cv.rectangle(seg_check, (row.x1, row.y1), (row.x2, row.y2), (255,0,0), 1)
            cv.putText(seg_check, str(idx), (row.x1+15, row.y1+15),
                        cv.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1, cv.LINE_AA)
        ret = cv.imwrite(vidPath[:-4] + '_segmentation.jpg', seg_check)

    ## Segment video and write frame to that videoWriter
    for idx, row in pl.iterrows():
        # Create copy of img
        img_tmp = resized.copy()

        # Crop image
        img_crop = crop(img_tmp, row.x1, row.y1, row.x2, row.y2)
        h,w = img_crop.shape[:2]

        # Write image
        globals()['vW{}'.format(idx)].write(img_crop)

# Release videoWriters
for vw_id in range(34):
    globals()['vW{}'.format(vw_id)].release()

# Release main video file
vid.release()



Splitting into separate videos and cropping.

